In [1]:
import logging
import sys

sys.path.insert(0, "Singer")
sys.path.insert(0, "research")
sys.path

['research',
 'Singer',
 '/Users/rebelraider/.pyenv/versions/3.9.20/lib/python39.zip',
 '/Users/rebelraider/.pyenv/versions/3.9.20/lib/python3.9',
 '/Users/rebelraider/.pyenv/versions/3.9.20/lib/python3.9/lib-dynload',
 '',
 '/Users/rebelraider/Documents/Python projects/Hackatons/XLabs-Hack-2024/.venv/lib/python3.9/site-packages']

In [2]:
from huggingface_hub import snapshot_download 
snapshot_download(repo_id="Cyanbox/Prompt-Singer")

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

'/Users/rebelraider/.cache/huggingface/hub/models--Cyanbox--Prompt-Singer/snapshots/4a9ad215081865e168df26ea4a54cc78e87378a6'

In [1]:
import sys

sys.path.insert(0, "Singer")
sys.path.insert(0, "research")

import torch
import Singer
import research
import fairseq
from fairseq.checkpoint_utils import load_model_ensemble_and_task
from fairseq.models.text_to_speech.hub_interface import TTSHubInterface

def generate_voice(text, model_path, data_cfg_path):
    """
    Generates speech from input text using a pre-trained TTS model.

    Args:
        text (str): The input text to be converted to speech.
        model_path (str): Path to the pre-trained TTS model checkpoint (.pt file).
        data_cfg_path (str): Path to the data configuration file (data.yaml).

    Returns:
        tuple: A tuple containing the waveform tensor and the sample rate.
    """
    # Load the pre-trained TTS model and task
    models, cfg, task = load_model_ensemble_and_task(
        [model_path],
        arg_overrides={"data_config": data_cfg_path}
    )
    model = models[0]
    print("cfg:", cfg)
    inference = TTSHubInterface(cfg, task, model)
    # Build the generator for inference
    generator = task.build_generator([model], cfg)

    # Prepare the input text
    text_inputs = text.strip()
    inputs = inference.get_model_input(task, text_inputs)

    # Generate the speech waveform
    with torch.no_grad():
        waveform, sample_rate = inference.get_prediction(task, model, generator, inputs)

    return waveform, sample_rate

# Example usage:
# Replace 'path/to/tts_model.pt' and 'path/to/data.yaml' with your actual paths.
text_to_speak = "Hello, this is a test."
waveform, sr = generate_voice(text_to_speak, '/Users/rebelraider/.cache/huggingface/hub/models--Cyanbox--Prompt-Singer/snapshots/4a9ad215081865e168df26ea4a54cc78e87378a6/prompt-singer-flant5-large-finetuned/checkpoint_last.pt', 'huita.yaml')

# You can then save the waveform to an audio file using a library like librosa or soundfile.
# For example:
import soundfile as sf
sf.write('output.wav', waveform.numpy(), sr)

/Users/rebelraider/Documents/Python projects/Hackatons/XLabs-Hack-2024/.venv/lib/python3.9/site-packages/fairscale/experimental/nn/offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
/Users/rebelraider/Documents/Python projects/Hackatons/XLabs-Hack-2024/.venv/lib/python3.9/site-packages/fairscale/experimental/nn/offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(orig_func)  # type: ignore
/Users/rebelraider/Documents/Python projects/Hackatons/XLabs-Hack-2024/Singer/fairseq/file_io.py:234: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which wi

FileNotFoundError: config.yaml not found

In [ ]:
import sys

sys.path.insert(0, "Singer")
sys.path.insert(0, "research")

import torch
import logging
from fairseq import checkpoint_utils, tasks, utils
from fairseq.dataclass.utils import convert_namespace_to_omegaconf
from omegaconf import DictConfig

def generate_voice(text_input: str, model_path: str):
    """
    Generate voice from text input using the specified model.

    Args:
        text_input (str): The input text to convert to voice.
        model_path (str): Path to the model checkpoint.

    Returns:
        output_text (str): The generated output text.
    """
    # Set up logging
    logging.basicConfig(
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        level=logging.INFO,
    )
    logger = logging.getLogger("generate_voice")

    # Prepare configuration
    cfg = DictConfig({
        'common': {
            'fp16': True,
            'cpu': False,
            'seed': None,
            'user_dir': 'research',
            'log_format': None,
            'log_interval': None,
            'no_progress_bar': False,
        },
        'common_eval': {
            'path': model_path,
            'results_path': None,
            'model_overrides': '{}',
            'post_process': 'sentencepiece',
            'quiet': True,
        },
        'dataset': {
            'max_tokens': 10000,
            'batch_size': 1,
            'gen_subset': 'test',
            'num_workers': 0,
            'data_buffer_size': 0,
            'skip_invalid_size_inputs_valid_test': False,
            'required_batch_size_multiple': 1,
            'dataset_impl': 'raw',
        },
        'generation': {
            'max_len_a': 1,
            'max_len_b': 0,
            'beam': 1,
            'sampling': False,
            'nbest': 1,
            'replace_unk': None,
            'no_seed_provided': False,
            'prefix_size': 0,
            'lm_path': None,
            'lm_weight': 0.0,
        },
        'distributed_training': {
            'distributed_world_size': 1,
            'distributed_rank': 0,
            'ddp_backend': 'no_c10d',
            'pipeline_model_parallel': False,
        },
        'task': {
            '_name': 't2a_sing_t5_config_task',  # Replace with your task name
            'data': '',  # Not used since we're providing data directly
        },
        'bpe': None,
        'tokenizer': None,
        'scoring': {
            '_name': 'bleu',
        },
    })

    # Import user modules if any
    utils.import_user_module(cfg.common)

    # Set up the task
    task = tasks.setup_task(cfg.task)

    # Load the model
    overrides = {}
    logger.info(f"Loading model from {cfg.common_eval.path}")
    models, _ = checkpoint_utils.load_model_ensemble(
        [cfg.common_eval.path],
        arg_overrides=overrides,
        task=task,
    )

    # Prepare the input sample
    src_dict = getattr(task, "source_dictionary", None)
    tgt_dict = task.target_dictionary
    if src_dict is None:
        logger.error("Source dictionary not found.")
        return None

    src_tokens = src_dict.encode_line(text_input, add_if_not_exist=False).long()
    src_lengths = torch.LongTensor([src_tokens.numel()])
    sample = {
        'net_input': {
            'src_tokens': src_tokens.unsqueeze(0),
            'src_lengths': src_lengths,
        }
    }

    # Move models and sample to GPU if available
    use_cuda = torch.cuda.is_available() and not cfg.common.cpu
    if use_cuda:
        sample = utils.move_to_cuda(sample)
        for model in models:
            model.cuda()
            if cfg.common.fp16:
                model.half()
    else:
        for model in models:
            if cfg.common.fp16:
                model.half()

    # Build the generator
    generator = task.build_generator(models, cfg.generation)

    # Run inference
    with torch.no_grad():
        hypos = task.inference_step(generator, models, sample)

    # Process the output
    hypo = hypos[0][0]  # First sentence, first hypothesis
    hypo_tokens = hypo['tokens']

    # Convert tokens to string
    hypo_str = tgt_dict.string(
        hypo_tokens,
        cfg.common_eval.post_process,
        extra_symbols_to_ignore={generator.eos}
    )

    # Decode using BPE and tokenizer if available
    tokenizer = task.build_tokenizer(cfg.tokenizer)
    bpe = task.build_bpe(cfg.bpe)

    def decode_fn(x):
        if bpe is not None:
            x = bpe.decode(x)
        if tokenizer is not None:
            x = tokenizer.decode(x)
        return x

    output_text = decode_fn(hypo_str)
    logger.info(f"Generated output: {output_text}")

    return output_text